# Day 9: Autonomous Agents & Self-Reflection 🪞

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**What you'll build today:**
1. Self-Reflection Loop — agent critiques and improves its own output
2. Plan-Execute Agent — breaks complex tasks into sub-tasks and works through them
3. ReWOO Agent — plans all tool calls upfront, executes once, synthesises
4. Combined: Plan + Reflect — each execution step has its own quality check
5. Mini-Project: Autonomous Research Agent — full pipeline

**Time:** ~2 hours | **Difficulty:** Beginner-friendly | **Prerequisites:** Day 8 complete

---

## 0. Setup

In [ ]:
# ============================================================
# COLAB USERS ONLY — Uncomment if using Google Colab
# LOCAL users: SKIP this cell
# ============================================================
# !pip install langgraph langchain-groq langchain-core python-dotenv ddgs -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

# One LLM used throughout the entire notebook
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

response = llm.invoke("Say 'Day 9 ready!' in exactly 3 words.")
print(response.content)

---
## 1. 🪞 Self-Reflection Loop

**The idea:** The agent produces an answer, then critiques it, then improves it. This loop repeats until the critique says "no issues" — or until 3 attempts (safety valve).

```
[generate] → [critique] → [improve] → [check_quality]
                                            ↓ still issues → back to [critique]
                                            ↓ no issues   → END
```

**Key insight:** Generating and evaluating are different tasks for an LLM. The critique prompt puts the LLM in "fresh eyes" mode — it spots issues it missed while generating.

In [ ]:
# ── State ──────────────────────────────────────────────────
class ReflectState(TypedDict):
    question:   str
    answer:     str
    critique:   str
    iterations: int   # safety valve counter


# ── Node 1: Generate ───────────────────────────────────────
def generate_node(state: ReflectState) -> dict:
    """Produces an initial answer."""
    question = state["question"]
    iterations = state.get("iterations", 0)
    previous_critique = state.get("critique", "")

    if iterations == 0:
        prompt = f"Answer this question clearly in 3-4 sentences: {question}"
        print(f"  [generate] Writing first answer...")
    else:
        previous_answer = state.get("answer", "")
        prompt = f"""Your previous answer had these issues:
{previous_critique}

Previous answer: {previous_answer}

Rewrite the answer to fix ALL the issues listed. Keep it 3-4 sentences."""
        print(f"  [generate] Rewriting (attempt {iterations + 1})...")

    response = llm.invoke(prompt)
    return {"answer": response.content, "iterations": iterations + 1}


# ── Node 2: Critique ───────────────────────────────────────
def critique_node(state: ReflectState) -> dict:
    """LLM evaluates the answer and lists specific issues."""
    question = state["question"]
    answer   = state["answer"]

    prompt = f"""You are a strict quality reviewer.

Question: {question}
Answer: {answer}

Review the answer against these criteria:
1. Does it directly answer the question?
2. Does it include at least one specific fact or example?
3. Is it clear and easy to understand?

If ALL criteria are met, reply with exactly: NO ISSUES
If any criterion is missing, list the specific issues (be brief)."""

    response = llm.invoke(prompt)
    critique = response.content.strip()
    print(f"  [critique] Result: {critique[:80]}")
    return {"critique": critique}


# ── Routing: should we improve or stop? ───────────────────
def check_quality(state: ReflectState) -> str:
    # Safety valve: stop after 3 attempts regardless
    if state.get("iterations", 0) >= 3:
        print(f"  [check_quality] Max iterations reached — stopping")
        return "end"

    critique = state.get("critique", "")
    if "NO ISSUES" in critique.upper():
        print(f"  [check_quality] Quality approved!")
        return "end"
    else:
        print(f"  [check_quality] Issues found — looping back")
        return "generate"   # loop back


# ── Build the graph ────────────────────────────────────────
reflect_graph = StateGraph(ReflectState)

reflect_graph.add_node("generate", generate_node)
reflect_graph.add_node("critique", critique_node)

reflect_graph.set_entry_point("generate")
reflect_graph.add_edge("generate", "critique")

reflect_graph.add_conditional_edges(
    "critique",
    check_quality,
    {"generate": "generate", "end": END}
)

reflect_app = reflect_graph.compile()
print("Self-reflection graph compiled!")

In [ ]:
# Test 1 — a question that likely needs improvement
print("=" * 55)
print("Question: How do neural networks learn?")
print("=" * 55)

result = reflect_app.invoke({"question": "How do neural networks learn?"})

print("=" * 55)
print(f"Total iterations: {result['iterations']}")
print(f"Final critique:   {result['critique']}")
print(f"\n=== Final Answer ===")
print(result["answer"])

In [ ]:
# Test 2 — try your own question
print("=" * 55)
print("Question: What is the difference between AI and ML?")
print("=" * 55)

result2 = reflect_app.invoke({"question": "What is the difference between AI and ML?"})

print(f"\nIterations needed: {result2['iterations']}")
print(f"\n=== Final Answer ===")
print(result2["answer"])

**What you just saw:**
- The LLM critiques its own output in a different "mode" than it generated it
- The conditional edge `check_quality` reads the critique and decides: loop or stop
- The safety valve (`iterations >= 3`) prevents infinite loops
- The `iterations` counter in state is what makes the safety valve work

---
## 2. 🗺️ Plan-Execute Agent

**The idea:** For complex tasks, plan all sub-tasks first, then execute them one by one, storing results in state. Finally, synthesise everything.

```
[planner] → [executor] → [check_done]
                              ↓ more tasks → back to [executor]
                              ↓ all done  → [synthesise] → END
```

**Why this beats one-shot:** A complex question sent directly to an LLM gets a shallow answer. Breaking it into sub-tasks forces depth on each part.

In [ ]:
import json

# ── State ──────────────────────────────────────────────────
class PlanState(TypedDict):
    task:          str
    plan:          List[str]   # list of sub-tasks created by planner
    current_step:  int         # which sub-task we're on
    results:       List[str]   # results from each completed sub-task
    final_answer:  str


# ── Node 1: Planner ────────────────────────────────────────
def planner_node(state: PlanState) -> dict:
    """Breaks the task into 3-5 specific sub-tasks."""
    task = state["task"]

    prompt = f"""Break this task into 3-5 specific, actionable sub-tasks:
Task: {task}

Return ONLY a JSON array of strings. Example:
["sub-task 1", "sub-task 2", "sub-task 3"]

No explanation, no markdown — just the JSON array."""

    response = llm.invoke(prompt)
    raw = response.content.strip()

    # Parse the plan
    try:
        # Strip markdown fences if present
        if "```" in raw:
            raw = raw.split("```")[1].replace("json", "").strip()
        plan = json.loads(raw)
    except Exception:
        # Fallback: split by newlines if JSON parse fails
        plan = [line.strip().lstrip("0123456789.-) ") for line in raw.split("\n") if line.strip()]

    print(f"  [planner] Created {len(plan)} sub-tasks:")
    for i, step in enumerate(plan):
        print(f"    {i+1}. {step}")

    return {"plan": plan, "current_step": 0, "results": []}


# ── Node 2: Executor ───────────────────────────────────────
def executor_node(state: PlanState) -> dict:
    """Executes the current sub-task and stores the result."""
    plan         = state["plan"]
    current_step = state["current_step"]
    results      = state.get("results", [])
    sub_task     = plan[current_step]

    print(f"  [executor] Step {current_step + 1}/{len(plan)}: {sub_task}")

    # Build context from previous results
    context = ""
    if results:
        context = "\n\nPrevious steps completed:\n" + "\n".join(
            f"Step {i+1}: {r[:150]}" for i, r in enumerate(results)
        )

    prompt = f"""You are working on this overall task: {state['task']}
{context}

Now complete this specific sub-task:
{sub_task}

Be specific and detailed (3-5 sentences)."""

    response = llm.invoke(prompt)
    new_results = results + [response.content]

    return {"results": new_results, "current_step": current_step + 1}


# ── Node 3: Synthesise ─────────────────────────────────────
def synthesise_node(state: PlanState) -> dict:
    """Combines all sub-task results into one coherent final answer."""
    task    = state["task"]
    plan    = state["plan"]
    results = state["results"]

    combined = "\n\n".join(
        f"Sub-task {i+1} ({plan[i]}):\n{result}"
        for i, result in enumerate(results)
    )

    prompt = f"""You completed a multi-step task. Here are all the results:

Overall task: {task}

{combined}

Now write a clear, coherent final answer that combines all these findings.
Use headings for each section. Aim for 200-300 words."""

    response = llm.invoke(prompt)
    print(f"  [synthesise] Final answer assembled")
    return {"final_answer": response.content}


# ── Routing: more tasks or done? ───────────────────────────
def check_done(state: PlanState) -> str:
    if state["current_step"] >= len(state["plan"]):
        return "synthesise"
    return "executor"


# ── Build the graph ────────────────────────────────────────
plan_graph = StateGraph(PlanState)

plan_graph.add_node("planner",    planner_node)
plan_graph.add_node("executor",   executor_node)
plan_graph.add_node("synthesise", synthesise_node)

plan_graph.set_entry_point("planner")
plan_graph.add_edge("planner", "executor")

plan_graph.add_conditional_edges(
    "executor",
    check_done,
    {"executor": "executor", "synthesise": "synthesise"}
)

plan_graph.add_edge("synthesise", END)

plan_app = plan_graph.compile()
print("Plan-Execute graph compiled!")

In [ ]:
print("=" * 60)
print("Task: Explain LangGraph — what it is, why it was built,")
print("      what problems it solves, and when to use it.")
print("=" * 60)

result = plan_app.invoke({
    "task": "Explain LangGraph — what it is, why it was built, what problems it solves, and when to use it over LangChain."
})

print("\n" + "=" * 60)
print("FINAL SYNTHESISED ANSWER")
print("=" * 60)
print(result["final_answer"])

**Notice the difference from one-shot:** Compare this answer to what you'd get from `llm.invoke("Explain LangGraph...")`. The Plan-Execute version is deeper and more structured because each sub-task was answered with full focus.

---
## 3. ⚡ ReWOO Agent — Always 2 LLM Calls

**The idea:** Instead of Think → Act → Observe → Think → Act (many LLM calls), ReWOO does:
1. **Reason once** — plan ALL tool calls upfront in one LLM call
2. **Execute all tools** — no LLM involved
3. **Synthesise once** — combine all results in one final LLM call

For any task with N steps: **always exactly 2 LLM calls** regardless of N.

In [ ]:
import datetime

# ── Mock tools (replace with real ones in production) ─────
def tool_calculate(expression: str) -> str:
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

def tool_get_date(query: str) -> str:
    today = datetime.date.today()
    return f"Today is {today.strftime('%B %d, %Y')} ({today.strftime('%A')})"

def tool_word_count(text: str) -> str:
    words = len(text.split())
    return f"Word count: {words} words"

def tool_search(query: str) -> str:
    # Simulated search — replace with DuckDuckGo in real use
    return f"[Simulated search result for: '{query}'] — LangGraph is a library for building stateful, multi-actor applications with LLMs using graph-based workflows."

# Tool registry
TOOLS = {
    "calculate":  tool_calculate,
    "get_date":   tool_get_date,
    "word_count": tool_word_count,
    "search":     tool_search,
}

print("Tools available:", list(TOOLS.keys()))

In [ ]:
# ── State ──────────────────────────────────────────────────
class ReWOOState(TypedDict):
    task:          str
    tool_plan:     List[dict]  # [{"tool": "name", "input": "...", "reason": "..."}]
    tool_results:  List[str]   # results from each tool call
    final_answer:  str


# ── Phase 1: Reason — plan ALL tool calls in ONE LLM call ─
def reason_node(state: ReWOOState) -> dict:
    """Plans all tool calls upfront. This is the ONLY LLM call before execution."""
    task = state["task"]

    prompt = f"""You have access to these tools:
- calculate: evaluates a math expression, input is the expression string
- get_date: returns today's date, input is any string (e.g. 'today')
- word_count: counts words in text, input is the text to count
- search: searches for information, input is the search query

Task: {task}

Plan ALL the tool calls needed to solve this task.
Return ONLY a JSON array. Each item must have: tool, input, reason.

Example format:
[
  {{"tool": "search", "input": "LangGraph overview", "reason": "need background info"}},
  {{"tool": "calculate", "input": "2 + 2", "reason": "verify the sum"}}
]

No explanation, no markdown fences — just the JSON array."""

    response = llm.invoke(prompt)
    raw = response.content.strip()

    try:
        if "```" in raw:
            raw = raw.split("```")[1].replace("json", "").strip()
        tool_plan = json.loads(raw)
    except Exception:
        # Fallback to single search if parsing fails
        tool_plan = [{"tool": "search", "input": task, "reason": "general lookup"}]

    print(f"  [reason] LLM call #1 — Planned {len(tool_plan)} tool call(s):")
    for step in tool_plan:
        print(f"    → {step.get('tool','?')}({step.get('input','?')[:50]}) — {step.get('reason','?')}")

    return {"tool_plan": tool_plan, "tool_results": []}


# ── Phase 2: Execute — run ALL tools (NO LLM calls) ───────
def execute_node(state: ReWOOState) -> dict:
    """Runs every planned tool call. Zero LLM calls in this phase."""
    tool_plan = state["tool_plan"]
    results   = []

    print(f"  [execute] Running {len(tool_plan)} tool(s) — no LLM calls")
    for step in tool_plan:
        tool_name  = step.get("tool", "search")
        tool_input = step.get("input", "")

        if tool_name in TOOLS:
            result = TOOLS[tool_name](tool_input)
        else:
            result = f"Unknown tool: {tool_name}"

        print(f"    ✓ {tool_name}: {result[:80]}")
        results.append(f"[{tool_name}({tool_input[:40]})] → {result}")

    return {"tool_results": results}


# ── Phase 3: Synthesise — one final LLM call ──────────────
def synthesise_rewoo(state: ReWOOState) -> dict:
    """Combines all tool results into a final answer. This is LLM call #2."""
    task         = state["task"]
    tool_results = state["tool_results"]

    results_text = "\n".join(tool_results)

    prompt = f"""Task: {task}

Tool results:
{results_text}

Using these results, write a clear and complete answer to the task."""

    response = llm.invoke(prompt)
    print(f"  [synthesise] LLM call #2 — Final answer written")
    return {"final_answer": response.content}


# ── Build the graph ────────────────────────────────────────
rewoo_graph = StateGraph(ReWOOState)

rewoo_graph.add_node("reason",     reason_node)
rewoo_graph.add_node("execute",    execute_node)
rewoo_graph.add_node("synthesise", synthesise_rewoo)

rewoo_graph.set_entry_point("reason")
rewoo_graph.add_edge("reason",     "execute")
rewoo_graph.add_edge("execute",    "synthesise")
rewoo_graph.add_edge("synthesise", END)

rewoo_app = rewoo_graph.compile()
print("ReWOO graph compiled! Always uses exactly 2 LLM calls.")

In [ ]:
print("=" * 60)
print("Task: What day of the week is today, and what is 365 * 24?")
print("=" * 60)

result = rewoo_app.invoke({
    "task": "What day of the week is today, and what is 365 multiplied by 24?"
})

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result["final_answer"])

In [ ]:
# Count the LLM calls — should always be 2
print("=" * 60)
print("Task: Search for what LangGraph is, then count the words")
print("      in this sentence: 'Agentic AI is the future'")
print("=" * 60)

result2 = rewoo_app.invoke({
    "task": "Search for what LangGraph is, and also count the words in: 'Agentic AI is the future of software engineering'"
})

print("\n" + "=" * 60)
print("Tool plan used:", [s["tool"] for s in result2["tool_plan"]])
print("LLM calls made: exactly 2 (reason + synthesise)")
print("\nFINAL ANSWER")
print(result2["final_answer"])

**The key difference from ReAct (Day 2):**
- ReAct: each tool call requires an LLM call before it → N tools = 2N+ LLM calls
- ReWOO: plan everything once, execute everything, synthesise once → always 2 LLM calls

**The tradeoff:** ReWOO can't adapt if a tool result changes the plan. ReAct can. Use ReWOO when you know the steps in advance.

---
## 4. 🔗 Combined: Plan + Reflect

**The most powerful pattern.** Plan the task, then for EACH sub-task: execute it AND reflect on whether the result is good enough before moving on.

```
[planner] → [executor] → [reflect_step] 
                               ↓ step needs redo → back to [executor] (same step)
                               ↓ step OK → check if all done
                                               ↓ more steps → [executor] (next step)
                                               ↓ all done   → [synthesise] → END
```

In [ ]:
# ── State ──────────────────────────────────────────────────
class PlanReflectState(TypedDict):
    task:           str
    plan:           List[str]
    current_step:   int
    step_result:    str     # result of the CURRENT step only
    step_critique:  str     # critique of the current step
    step_retries:   int     # retries for current step
    all_results:    List[str]
    final_answer:   str


# ── Planner (same as before) ───────────────────────────────
def pr_planner(state: PlanReflectState) -> dict:
    task = state["task"]
    prompt = f"""Break this task into 3 specific sub-tasks:
Task: {task}
Return ONLY a JSON array of 3 strings. No explanation."""
    response = llm.invoke(prompt)
    raw = response.content.strip()
    try:
        if "```" in raw:
            raw = raw.split("```")[1].replace("json","").strip()
        plan = json.loads(raw)
    except Exception:
        plan = [line.strip() for line in raw.split("\n") if line.strip()][:3]
    print(f"  [planner] Plan: {plan}")
    return {"plan": plan, "current_step": 0, "all_results": [], "step_retries": 0}


# ── Executor: run the CURRENT step only ───────────────────
def pr_executor(state: PlanReflectState) -> dict:
    plan         = state["plan"]
    current_step = state["current_step"]
    step_retries = state.get("step_retries", 0)
    sub_task     = plan[current_step]
    step_critique= state.get("step_critique", "")

    print(f"  [executor] Step {current_step+1}/{len(plan)} (retry #{step_retries}): {sub_task[:50]}")

    if step_retries > 0 and step_critique:
        prompt = f"""Redo this sub-task and fix the issue: {step_critique}
Sub-task: {sub_task}
Previous attempt: {state.get('step_result','')}
Write an improved response (3-4 sentences)."""
    else:
        prompt = f"""Complete this sub-task clearly in 3-4 sentences:
Overall goal: {state['task']}
Sub-task: {sub_task}"""

    response = llm.invoke(prompt)
    return {"step_result": response.content, "step_retries": step_retries + 1}


# ── Reflect: was this step good enough? ───────────────────
def pr_reflect(state: PlanReflectState) -> dict:
    sub_task    = state["plan"][state["current_step"]]
    step_result = state["step_result"]

    prompt = f"""Review this response to the sub-task.
Sub-task: {sub_task}
Response: {step_result}

Is the response specific and detailed (not vague)?
Reply with 'GOOD' if it is acceptable, or 'REDO: <reason>' if not."""

    response = llm.invoke(prompt)
    critique = response.content.strip()
    print(f"  [reflect] Step {state['current_step']+1}: {critique[:60]}")
    return {"step_critique": critique}


# ── Routing: redo step, next step, or done? ────────────────
def pr_route(state: PlanReflectState) -> str:
    critique     = state.get("step_critique", "")
    step_retries = state.get("step_retries", 0)

    # If step needs redo AND we haven't retried too many times
    if critique.upper().startswith("REDO") and step_retries < 2:
        return "executor"  # redo this step

    # Step is done — commit the result and move on
    all_results  = state.get("all_results", []) + [state["step_result"]]
    next_step    = state["current_step"] + 1

    # We can't update state from the routing function directly,
    # so we use a helper node below
    return "commit_step"


# ── Commit: save step result, advance counter ──────────────
def pr_commit(state: PlanReflectState) -> dict:
    all_results = state.get("all_results", []) + [state["step_result"]]
    next_step   = state["current_step"] + 1
    print(f"  [commit] Step {state['current_step']+1} committed. Moving to step {next_step+1}.")
    return {"all_results": all_results, "current_step": next_step,
            "step_retries": 0, "step_critique": ""}


# ── Check if all steps done ────────────────────────────────
def pr_check_done(state: PlanReflectState) -> str:
    if state["current_step"] >= len(state["plan"]):
        return "synthesise"
    return "executor"


# ── Synthesise ─────────────────────────────────────────────
def pr_synthesise(state: PlanReflectState) -> dict:
    plan    = state["plan"]
    results = state["all_results"]
    combined = "\n\n".join(f"Step {i+1} — {plan[i]}:\n{r}" for i, r in enumerate(results))
    prompt = f"""Combine these into a final answer for: {state['task']}

{combined}

Write a cohesive response using headings."""
    response = llm.invoke(prompt)
    print("  [synthesise] Done")
    return {"final_answer": response.content}


# ── Build the graph ────────────────────────────────────────
pr_graph = StateGraph(PlanReflectState)

pr_graph.add_node("planner",     pr_planner)
pr_graph.add_node("executor",    pr_executor)
pr_graph.add_node("reflect",     pr_reflect)
pr_graph.add_node("commit_step", pr_commit)
pr_graph.add_node("synthesise",  pr_synthesise)

pr_graph.set_entry_point("planner")
pr_graph.add_edge("planner",  "executor")
pr_graph.add_edge("executor", "reflect")

pr_graph.add_conditional_edges(
    "reflect", pr_route,
    {"executor": "executor", "commit_step": "commit_step"}
)

pr_graph.add_conditional_edges(
    "commit_step", pr_check_done,
    {"executor": "executor", "synthesise": "synthesise"}
)

pr_graph.add_edge("synthesise", END)

pr_app = pr_graph.compile()
print("Plan + Reflect graph compiled!")

In [ ]:
print("=" * 60)
print("Task: Explain what makes a good AI agent system design")
print("=" * 60)

result = pr_app.invoke({
    "task": "Explain what makes a good AI agent system design — covering architecture, reliability, and scalability"
})

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result["final_answer"])

---
## 5. 🏆 Mini-Project: Autonomous Research Agent

Putting it all together. This agent:
1. **Plans** the research sub-tasks
2. **Executes** each with web search (DuckDuckGo)
3. **Reflects** on each search result
4. **Synthesises** a structured research report

This is as close to a production autonomous research agent as you can build in a notebook.

In [ ]:
# Web search tool using ddgs
try:
    from ddgs import DDGS
    HAS_SEARCH = True
    print("✅ DuckDuckGo search available")
except ImportError:
    HAS_SEARCH = False
    print("⚠️  ddgs not installed — using simulated search")
    print("   Install with: pip install ddgs")

def web_search(query: str, max_results: int = 3) -> str:
    """Search the web and return a summary of top results."""
    if not HAS_SEARCH:
        return f"[Simulated] Search result for '{query}': Found relevant information about {query}. Key facts include recent developments and industry trends."
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return f"No results found for: {query}"
        summary = f"Search results for '{query}':\n"
        for i, r in enumerate(results):
            summary += f"\n[{i+1}] {r.get('title','')}: {r.get('body','')[:200]}"
        return summary
    except Exception as e:
        return f"Search error: {e}"

# Quick test
print(web_search("LangGraph agent framework", max_results=2)[:300])

In [ ]:
# ── State ──────────────────────────────────────────────────
class ResearchState(TypedDict):
    topic:        str
    plan:         List[str]   # research sub-questions
    current_step: int
    search_results: List[str]
    step_retries: int
    final_report: str


def research_planner(state: ResearchState) -> dict:
    """Creates 3 specific research sub-questions for the topic."""
    topic = state["topic"]
    prompt = f"""Create 3 specific research questions to thoroughly cover this topic:
Topic: {topic}

Each question should target a different angle (e.g. what it is, how it works, real-world use).
Return ONLY a JSON array of 3 strings."""
    response = llm.invoke(prompt)
    raw = response.content.strip()
    try:
        if "```" in raw:
            raw = raw.split("```")[1].replace("json","").strip()
        plan = json.loads(raw)
    except Exception:
        plan = [f"What is {topic}?", f"How does {topic} work?", f"Real-world applications of {topic}"]
    print(f"  [planner] Research questions:")
    for i, q in enumerate(plan):
        print(f"    {i+1}. {q}")
    return {"plan": plan, "current_step": 0, "search_results": [], "step_retries": 0}


def research_executor(state: ResearchState) -> dict:
    """Searches the web for the current research question."""
    question = state["plan"][state["current_step"]]
    print(f"  [executor] Searching: {question[:60]}")
    result = web_search(question, max_results=3)
    return {"search_results": state.get("search_results", []) + [result],
            "step_retries": state.get("step_retries", 0) + 1}


def research_reflect(state: ResearchState) -> dict:
    """Checks if the search result is useful. If poor, will re-search."""
    question   = state["plan"][state["current_step"]]
    last_result= state["search_results"][-1]
    prompt = f"""Does this search result usefully answer the question?
Question: {question}
Result: {last_result[:400]}

Reply 'GOOD' if it contains useful information, or 'REDO' if it is empty or irrelevant."""
    response = llm.invoke(prompt)
    verdict  = response.content.strip().upper()
    print(f"  [reflect] Step {state['current_step']+1}: {'✅ GOOD' if 'GOOD' in verdict else '🔄 REDO'}")
    return {"step_retries": state.get("step_retries", 0)}


def research_route(state: ResearchState) -> str:
    last_result  = state["search_results"][-1] if state["search_results"] else ""
    step_retries = state.get("step_retries", 0)
    # If result is too short and we haven't retried, redo
    if len(last_result) < 100 and step_retries < 2:
        return "executor"
    # Otherwise advance to next step
    return "advance"


def research_advance(state: ResearchState) -> dict:
    """Move to the next research question."""
    return {"current_step": state["current_step"] + 1, "step_retries": 0}


def research_check_done(state: ResearchState) -> str:
    if state["current_step"] >= len(state["plan"]):
        return "report"
    return "executor"


def research_report(state: ResearchState) -> dict:
    """Writes the final research report from all search results."""
    topic   = state["topic"]
    plan    = state["plan"]
    results = state["search_results"]

    sections = "\n\n".join(
        f"Research Question {i+1}: {plan[i]}\nFindings: {results[i][:600]}"
        for i in range(min(len(plan), len(results)))
    )

    prompt = f"""Write a structured research report on: {topic}

Based on these research findings:
{sections}

Format the report with:
1. Executive Summary (2-3 sentences)
2. Key Findings (one section per research question)
3. Conclusion"""

    response = llm.invoke(prompt)
    print("  [report] Research report written")
    return {"final_report": response.content}


# ── Build the graph ────────────────────────────────────────
research_graph = StateGraph(ResearchState)

research_graph.add_node("planner",  research_planner)
research_graph.add_node("executor", research_executor)
research_graph.add_node("reflect",  research_reflect)
research_graph.add_node("advance",  research_advance)
research_graph.add_node("report",   research_report)

research_graph.set_entry_point("planner")
research_graph.add_edge("planner",  "executor")
research_graph.add_edge("executor", "reflect")

research_graph.add_conditional_edges(
    "reflect", research_route,
    {"executor": "executor", "advance": "advance"}
)

research_graph.add_conditional_edges(
    "advance", research_check_done,
    {"executor": "executor", "report": "report"}
)

research_graph.add_edge("report", END)

research_app = research_graph.compile()
print("Autonomous Research Agent compiled and ready!")

In [ ]:
print("=" * 60)
print("Research Topic: LangGraph for Production AI Systems")
print("=" * 60)

result = research_app.invoke({"topic": "LangGraph for production AI systems"})

print("\n" + "=" * 60)
print("RESEARCH REPORT")
print("=" * 60)
print(result["final_report"])

---
## 6. 🎯 Your Turn — Challenge Exercises

### Challenge 1 (Easy): Stricter self-reflection
Modify the self-reflection loop from Section 1 to also check that the answer includes at least one real-world example. Add this as a 4th criterion in the critique prompt.

In [ ]:
# YOUR CODE HERE
# Hint: In critique_node, add a 4th criterion to the prompt:
# "4. Does it include at least one real-world example or application?"
# Then test with: "What is transfer learning?"

### Challenge 2 (Medium): Add real web search to the Plan-Execute agent
Replace the simple LLM calls in `executor_node` (Section 2) with actual web search using `web_search()` from Section 5. The executor should search for each sub-task, then the synthesiser combines the real search results.

In [ ]:
# YOUR CODE HERE
# Hint: In executor_node, replace llm.invoke() with:
#   search_result = web_search(sub_task)
#   new_results = results + [search_result]
#   return {"results": new_results, "current_step": current_step + 1}

### Challenge 3 (Hard): Build a ReWOO agent with real tools
Add a real `web_search` tool to the ReWOO agent's tool registry (Section 3). The reasoner should use `search` as a tool option. Test it with: `"What is the latest version of LangGraph and what does it do?"`

In [ ]:
# YOUR CODE HERE
# Hint: In the TOOLS dict, replace the simulated tool_search with the real web_search:
#   TOOLS = {
#       "calculate":  tool_calculate,
#       "get_date":   tool_get_date,
#       "word_count": tool_word_count,
#       "search":     web_search,   # <-- real search from Section 5
#   }
# Then run rewoo_app.invoke({"task": "What is the latest version of LangGraph?"})

---
## 7. Day 9 Wrap-up

### What you built today:
- ✅ **Self-Reflection** — generate → critique → improve loop with safety valve
- ✅ **Plan-Execute** — structured multi-step task decomposition
- ✅ **ReWOO** — always 2 LLM calls regardless of task complexity
- ✅ **Plan + Reflect** — per-step quality checking in a complex workflow
- ✅ **Autonomous Research Agent** — full pipeline with web search and reflection

### The one rule to remember:
```
Every pattern = different arrangement of nodes and edges.
The LangGraph framework never changes.
```

### Safety valve rule (always required for loops):
```python
# ALWAYS include this in every loop
if state.get("iterations", 0) >= MAX_ITERATIONS:
    return "end"  # break out no matter what
```

### Tomorrow — Day 10: RAG + Memory
Combining retrieval (ChromaDB, from Day 4-5) with agent memory (from Day 3) inside a LangGraph workflow. An agent that remembers conversations AND retrieves from a knowledge base.

---
### 📚 Resources:
- [LangGraph tutorials](https://langchain-ai.github.io/langgraph/tutorials/)
- [ReWOO paper](https://arxiv.org/abs/2305.18323)
- [Reflexion paper](https://arxiv.org/abs/2303.11366) — the academic basis for self-reflection